# 🏡 Airbnb Listings — Deep Dive Analysis

**Dataset:** 953 Airbnb listings scraped from the platform  
**Goal:** Explore pricing, amenities, location patterns, and listing themes through structured feature extraction and statistical analysis.

---

## 📋 Notebook Structure

| Section | What Happens |
|---|---|
| **1 · Setup** | Imports and module loading |
| **2 · Raw Data Load** | Read CSV, inspect shape and column types |
| **3 · Initial Exploration** | Missing values, duplicates, unique counts |
| **4 · Sample Rows** | Visual sanity-check of raw data |
| **5 · Feature Extraction** | Parse `Title` and `Detail` → 22 new structured columns (7 → 29) |
| **6 · Data Cleaning** | Parse prices, ratings, bed info, dates → 13 more columns (29 → 42) |
| **7 · Column Audit** | Full list of all 42 columns with dtypes |
| **8 · Spot Check** | Explore a specific column in depth |
| **9 · Prepare for Analysis** | Drop redundant columns, reorder → clean 32-column DataFrame |
| **10 · Save to Parquet** | Persist the cleaned DataFrame to disk for future use |
| **11 · Reload from Parquet** | Fast entry point — skip the pipeline on subsequent runs |

> **⚡ Quick Start:** If `cleaned_and_processed_data.parquet` already exists, jump directly to **Section 11** and skip Sections 2–10.


## 1 · Setup

Import the standard libraries and all custom modules.  
Each module is responsible for a distinct stage of the pipeline:

- `data_loader.py` — CSV loading and initial exploration helpers  
- `feature_extraction.py` — Regex/keyword-based column creation from `Title` and `Detail`  
- `data_cleaning.py` — Type conversion and parsing for prices, ratings, beds, and dates  
- `analysis.py` — Final column selection, parquet persistence, and (future) visualizations


## 2 · Raw Data Load

Load the raw CSV file. The dataset starts with **7 columns**, all stored as plain strings — no numeric types yet.


In [14]:
import pandas as pd
from data_loader import load_data, explore_data, show_sample, show_column_summary
from feature_extraction import extract_all_features
from data_cleaning import clean_all_columns


In [15]:
df = load_data()

[OK] Data loaded successfully: 953 rows, 7 columns


## 3 · Initial Data Exploration

Get a structural overview of the raw DataFrame:
- **Shape** — 953 rows × 7 columns
- **Dtypes & null counts** — everything is `str`; `Offer price` is 82.6% missing
- **Duplicates** — 34 duplicate rows (3.57%)
- **Unique values** — high cardinality in `Title`, `Detail`, and `Review and rating`


In [16]:
explore_data(df)


INITIAL DATA EXPLORATION

>> Shape: 953 rows x 7 columns

>> Data Types & Non-Null Counts:
----------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Title                   953 non-null    str  
 1   Detail                  953 non-null    str  
 2   Date                    953 non-null    str  
 3   Price(in dollar)        953 non-null    str  
 4   Offer price(in dollar)  166 non-null    str  
 5   Review and rating       947 non-null    str  
 6   Number of bed           953 non-null    str  
dtypes: str(7)
memory usage: 146.8 KB

>> Missing Values:
----------------------------------------
                        Missing  Percentage
Offer price(in dollar)      787       82.58
Review and rating             6        0.63

>> Duplicate Rows: 34 (3.57%)

>> Unique Values Per Column:
------------------------

## 4 · Sample Rows

A quick visual look at the first 5 rows to understand the raw string formats.

Key observations:
- `Title` encodes **property type + full location** in one string (e.g., *"Cabin in Hancock, New York, US"*)
- `Detail` is a short marketing tagline, not a full description
- `Date` is a range string (e.g., *"Jun 11 - 16"*)
- `Price(in dollar)` and `Offer price(in dollar)` are decimal strings, not floats
- `Review and rating` combines rating score and review count (e.g., *"4.85 (531)"*)


In [17]:
show_sample(df)



>> First 5 rows:


,Title,Detail,Date,Price(in dollar),Offer price(in dollar),Review and rating,Number of bed
0,"Chalet in Skykomish, Washington, US",Sky Haus - A-Frame Cabin,Jun 11 - 16,306.00,229.00,4.85 (531),4 beds
1,"Cabin in Hancock, New York, US",The Catskill A-Frame - Mid-Century Modern Cabin,Jun 6 - 11,485.00,170.00,4.77 (146),4 beds
2,"Cabin in West Farmington, Ohio, US",The Triangle: A-Frame Cabin for your city retreat,Jul 9 - 14,119.00,522.00,4.91 (515),4 beds
3,"Home in Blue Ridge, Georgia, US",*Summer Sizzle* 5 Min to Blue Ridge* Pets* Hot...,Jun 11 - 16,192.00,348.00,4.94 (88),5 beds
4,"Treehouse in Grandview, Texas, US",Luxury Treehouse Couples Getaway w/ Peaceful V...,Jun 4 - 9,232.00,196.00,4.99 (222),1 queen bed


## 5 · Feature Extraction  `(7 → 29 columns)`

Apply `extract_all_features()` from `feature_extraction.py`.  
This parses the unstructured `Title` and `Detail` columns using **regex patterns and keyword scoring** to produce 22 new analysis-ready columns:

| Group | New Columns |
|---|---|
| **From Title** | `Property_Type`, `City`, `State`, `Country` |
| **Amenity flags** | `Has_Hot_Tub`, `Has_Sauna`, `Has_Pool`, `Has_BBQ`, `Has_Fireplace`, `Has_Parking`, `Has_Spa` |
| **Location flags** | `Has_Waterfront`, `Is_Near_Beach`, `Is_Mountain`, `Is_Rural`, `Is_Urban` |
| **Classification** | `Property_Subtype`, `Theme` |
| **Text metrics** | `Detail_Word_Count`, `Detail_Char_Count`, `Has_Promo`, `Bedroom_Count` |


In [18]:
df = extract_all_features(df)
print("Shape after feature extraction:", df.shape)
df.head()


Shape after feature extraction: (953, 29)


,Title,Detail,Date,Price(in dollar),Offer price(in dollar),Review and rating,Number of bed,Property_Type,City,State,...,Is_Near_Beach,Is_Mountain,Is_Rural,Is_Urban,Property_Subtype,Theme,Detail_Word_Count,Detail_Char_Count,Has_Promo,Bedroom_Count
0,"Chalet in Skykomish, Washington, US",Sky Haus - A-Frame Cabin,Jun 11 - 16,306.00,229.00,4.85 (531),4 beds,Chalet,Skykomish,Washington,...,False,False,False,False,A-Frame,Nature,5,24,False,NaN
1,"Cabin in Hancock, New York, US",The Catskill A-Frame - Mid-Century Modern Cabin,Jun 6 - 11,485.00,170.00,4.77 (146),4 beds,Cabin,Hancock,New York,...,False,False,False,False,A-Frame,Nature,7,47,False,NaN
2,"Cabin in West Farmington, Ohio, US",The Triangle: A-Frame Cabin for your city retreat,Jul 9 - 14,119.00,522.00,4.91 (515),4 beds,Cabin,West Farmington,Ohio,...,False,False,True,False,A-Frame,Nature,8,49,False,NaN
3,"Home in Blue Ridge, Georgia, US",*Summer Sizzle* 5 Min to Blue Ridge* Pets* Hot...,Jun 11 - 16,192.00,348.00,4.94 (88),5 beds,Home,Blue Ridge,Georgia,...,False,True,False,False,Other,General,10,50,False,NaN
4,"Treehouse in Grandview, Texas, US",Luxury Treehouse Couples Getaway w/ Peaceful V...,Jun 4 - 9,232.00,196.00,4.99 (222),1 queen bed,Treehouse,Grandview,Texas,...,False,False,False,False,Treehouse,Luxury,7,50,False,NaN


## 6 · Data Cleaning  `(29 → 42 columns)`

Apply `clean_all_columns()` from `data_cleaning.py`.  
This converts the raw string columns into properly-typed numeric and categorical columns, adding 13 more:

| Source Column | New Columns Derived |
|---|---|
| `Price(in dollar)` | `Price` (float) |
| `Offer price(in dollar)` | `Offer_Price` (float), `Discount_Pct` (%) |
| `Review and rating` | `Rating` (float), `Review_Count` (int), `Is_New_Listing` (bool) |
| `Number of bed` | `Bed_Count` (int), `Bed_Type` (str) |
| `Date` | `Month`, `Start_Day`, `End_Day`, `Duration_Nights`, `Price_Per_Night` |


In [19]:
df = clean_all_columns(df)
print("Shape after cleaning:", df.shape)
df.head()


[OK] Data cleaning complete: 42 columns total
Shape after cleaning: (953, 42)


,Title,Detail,Date,Price(in dollar),Offer price(in dollar),Review and rating,Number of bed,Property_Type,City,State,...,Rating,Review_Count,Is_New_Listing,Bed_Count,Bed_Type,Month,Start_Day,End_Day,Duration_Nights,Price_Per_Night
0,"Chalet in Skykomish, Washington, US",Sky Haus - A-Frame Cabin,Jun 11 - 16,306.00,229.00,4.85 (531),4 beds,Chalet,Skykomish,Washington,...,4.85,531.0,False,4,Standard,June,11.0,16.0,5.0,61.2
1,"Cabin in Hancock, New York, US",The Catskill A-Frame - Mid-Century Modern Cabin,Jun 6 - 11,485.00,170.00,4.77 (146),4 beds,Cabin,Hancock,New York,...,4.77,146.0,False,4,Standard,June,6.0,11.0,5.0,97.0
2,"Cabin in West Farmington, Ohio, US",The Triangle: A-Frame Cabin for your city retreat,Jul 9 - 14,119.00,522.00,4.91 (515),4 beds,Cabin,West Farmington,Ohio,...,4.91,515.0,False,4,Standard,July,9.0,14.0,5.0,23.8
3,"Home in Blue Ridge, Georgia, US",*Summer Sizzle* 5 Min to Blue Ridge* Pets* Hot...,Jun 11 - 16,192.00,348.00,4.94 (88),5 beds,Home,Blue Ridge,Georgia,...,4.94,88.0,False,5,Standard,June,11.0,16.0,5.0,38.4
4,"Treehouse in Grandview, Texas, US",Luxury Treehouse Couples Getaway w/ Peaceful V...,Jun 4 - 9,232.00,196.00,4.99 (222),1 queen bed,Treehouse,Grandview,Texas,...,4.99,222.0,False,1,Queen,June,4.0,9.0,5.0,46.4


## 7 · Full Column Audit

Print all 42 columns with their datatypes and non-null counts to confirm the pipeline ran correctly.


In [20]:
print(f"Total columns: {df.shape[1]}\n")
for i, col in enumerate(df.columns, 1):
    dtype = df[col].dtype
    non_null = df[col].notna().sum()
    print(f"  {i:2d}. {col:<25s} {str(dtype):<10s} ({non_null} non-null)")


Total columns: 42

   1. Title                     str        (953 non-null)
   2. Detail                    str        (953 non-null)
   3. Date                      str        (953 non-null)
   4. Price(in dollar)          str        (953 non-null)
   5. Offer price(in dollar)    str        (166 non-null)
   6. Review and rating         str        (947 non-null)
   7. Number of bed             str        (953 non-null)
   8. Property_Type             str        (953 non-null)
   9. City                      str        (953 non-null)
  10. State                     str        (953 non-null)
  11. Country                   str        (953 non-null)
  12. Has_Hot_Tub               bool       (953 non-null)
  13. Has_Sauna                 bool       (953 non-null)
  14. Has_Pool                  bool       (953 non-null)
  15. Has_BBQ                   bool       (953 non-null)
  16. Has_Fireplace             bool       (953 non-null)
  17. Has_Parking               bool       (953 non-n

## 8 · Spot Check — Column Deep Dive

Use `show_column_summary()` to inspect any individual column.  
Currently profiling `Country` — **251 listings return "Unknown"** because many international listing titles don't follow the standard `"Type in City, State, Country"` format.

> Uncomment other columns below to inspect `Theme`, `Price`, or `Rating`.


In [21]:
show_column_summary(df, "Country")
# show_column_summary(df, "Theme")
# show_column_summary(df, "Price")
# show_column_summary(df, "Rating")



>> Column Summary: 'Country'
----------------------------------------
  Type:       str
  Non-Null:   953 / 953
  Null:       0
  Unique:     37

  Top 10 Values:
Country
Unknown        251
US             115
Indonesia      110
Thailand        69
Canada          44
Mexico          40
Philippines     37
Italy           34
UK              34
Malaysia        26


## 9 · Prepare Analysis DataFrame & Save to Parquet

`prepare_analysis_df()` drops the 10 redundant/low-value columns (the original 7 raw strings + `Bedroom_Count`, `Start_Day`, `End_Day`) and reorders the remaining **32 columns** into logical groups:

`Location → Pricing → Reviews → Beds → Dates → Amenities → Location Flags → Classification → Text Metrics`

The cleaned DataFrame is then saved to disk as a **Parquet file** using `fastparquet`, so this expensive pipeline doesn't need to run again.

> 📁 Output: `../data/cleaned_and_processed_data.parquet`


In [22]:
from analysis import prepare_analysis_df, save_analysis_df, load_analysis_df, parquet_exists

# First time — run full pipeline and save:
df = prepare_analysis_df(df)
save_analysis_df(df)


[OK] Analysis DataFrame ready: 953 rows × 32 columns
[OK] Saved analysis DataFrame → c:\Users\lenov\OneDrive\Desktop\ai-project-collection\Airbnb-Listings-Deep-Dive\data\cleaned_and_processed_data.parquet  (953 rows × 32 cols)


'c:\\Users\\lenov\\OneDrive\\Desktop\\ai-project-collection\\Airbnb-Listings-Deep-Dive\\data\\cleaned_and_processed_data.parquet'

## 10 · Reload from Parquet  *(Subsequent-Run Entry Point)*

---

> ### ⚡ Start here on all future runs
> 
> If `cleaned_and_processed_data.parquet` already exists, **run only this cell** instead of re-executing the full pipeline above.  
> It loads the clean, 32-column analysis-ready DataFrame directly from disk in milliseconds.

---


In [23]:
# Subsequent times — skip the pipeline, just load:
if parquet_exists():
    df = load_analysis_df()


[OK] Loaded analysis DataFrame ← c:\Users\lenov\OneDrive\Desktop\ai-project-collection\Airbnb-Listings-Deep-Dive\data\cleaned_and_processed_data.parquet  (953 rows × 32 cols)


## 11 · Verify the Loaded DataFrame

Confirm the final analysis DataFrame looks correct — 953 rows, 32 clean columns, all properly typed.

**From here, add analysis cells below** ↓  
*(Univariate → Bivariate → Multivariate)*


In [24]:
df.head()

,Property_Type,Property_Subtype,City,State,Country,Price,Offer_Price,Discount_Pct,Price_Per_Night,Rating,...,Has_Spa,Has_Waterfront,Is_Near_Beach,Is_Mountain,Is_Rural,Is_Urban,Theme,Detail_Word_Count,Detail_Char_Count,Has_Promo
0,Chalet,A-Frame,Skykomish,Washington,US,306.0,229.0,25.16,61.2,4.85,...,False,False,False,False,False,False,Nature,5,24,False
1,Cabin,A-Frame,Hancock,New York,US,485.0,170.0,64.95,97.0,4.77,...,False,False,False,False,False,False,Nature,7,47,False
2,Cabin,A-Frame,West Farmington,Ohio,US,119.0,522.0,-338.66,23.8,4.91,...,False,False,False,False,True,False,Nature,8,49,False
3,Home,Other,Blue Ridge,Georgia,US,192.0,348.0,-81.25,38.4,4.94,...,False,False,False,True,False,False,General,10,50,False
4,Treehouse,Treehouse,Grandview,Texas,US,232.0,196.0,15.52,46.4,4.99,...,False,False,False,False,False,False,Luxury,7,50,False
